# LeetCode 1524 — Number of Sub-arrays With Odd Sum bằng Python

Link đề: https://leetcode.com/problems/number-of-sub-arrays-with-odd-sum/ · Độ khó: **Medium** · Chủ đề: Array, Math, Dynamic Programming, Prefix Sum

Skeleton LeetCode:

```python
class Solution:
    def numOfSubarrays(self, arr: list[int]) -> int:
        ...
```

## 1. Mục tiêu học

1. Dùng **tổng tiền tố (prefix sum)** để biểu diễn tổng mọi mảng con: `sum(arr[i..j]) = P[j+1] − P[i]`.
2. Nhận ra chỉ cần **tính chẵn lẻ**: mảng con có tổng lẻ ⇔ hai prefix ở hai đầu **khác tính chẵn lẻ**.
3. Viết được 4 cách: brute force `O(n²)`, đếm prefix chẵn/lẻ một lượt `O(n)`, **DP** số mảng con kết thúc tại `i` có tổng lẻ/chẵn, và công thức đóng `E·O`.
4. Nhớ khởi tạo **prefix rỗng `P[0] = 0`** (chẵn) — lỗi kinh điển nhất của dạng bài này.
5. Lấy modulo `10⁹ + 7` đúng chỗ và tổng quát hoá sang "tổng chia hết cho k" (LC 974).

## 2. Đề bài & ví dụ

Cho mảng số nguyên `arr`, trả về số **mảng con** (liên tiếp, khác rỗng) có **tổng lẻ**. Kết quả có thể rất lớn → trả về modulo `10⁹ + 7`.

**Ví dụ 1:** `arr = [1,3,5]` → `4`. Các mảng con `[1],[1,3],[1,3,5],[3],[3,5],[5]` có tổng `1,4,9,3,8,5`; tổng lẻ: `1, 9, 3, 5`.

**Ví dụ 2:** `arr = [2,4,6]` → `0`.

**Ví dụ 3:** `arr = [1,2,3,4,5,6,7]` → `16`.

**Ràng buộc:** `1 ≤ len(arr) ≤ 10⁵`, `1 ≤ arr[i] ≤ 100`.

In [1]:
import random
import time
from collections import Counter
from itertools import accumulate
from typing import List

MOD = 10**9 + 7
examples = [([1, 3, 5], 4), ([2, 4, 6], 0), ([1, 2, 3, 4, 5, 6, 7], 16)]
for arr, want in examples:
    subs = [arr[i:j + 1] for i in range(len(arr)) for j in range(i, len(arr))]
    odd = [s for s in subs if sum(s) % 2]
    print(arr, "->", len(odd), "| ví dụ mảng con tổng lẻ:", odd[:4], "| kỳ vọng", want)

[1, 3, 5] -> 4 | ví dụ mảng con tổng lẻ: [[1], [1, 3, 5], [3], [5]] | kỳ vọng 4
[2, 4, 6] -> 0 | ví dụ mảng con tổng lẻ: [] | kỳ vọng 0
[1, 2, 3, 4, 5, 6, 7] -> 16 | ví dụ mảng con tổng lẻ: [[1], [1, 2], [1, 2, 3, 4, 5], [1, 2, 3, 4, 5, 6]] | kỳ vọng 16


## 3. Kiến thức nền

### 3.1 Tổng tiền tố

`P[0] = 0`, `P[k] = arr[0] + … + arr[k−1]`. Tổng `arr[i..j] = P[j+1] − P[i]`. `itertools.accumulate` tính nhanh các tổng chạy.

In [2]:
arr = [1, 2, 3, 4]
P = [0] + list(accumulate(arr))
print("P =", P)
i, j = 1, 3
print(f"sum(arr[{i}..{j}]) =", sum(arr[i:j + 1]), "= P[j+1] - P[i] =", P[j + 1] - P[i])

P = [0, 1, 3, 6, 10]
sum(arr[1..3]) = 9 = P[j+1] - P[i] = 9


### 3.2 Tính chẵn lẻ của hiệu

`a − b` lẻ ⇔ `a` và `b` khác tính chẵn lẻ. Trong Python `x % 2` cho 0 (chẵn) hoặc 1 (lẻ), kể cả với số âm. Có thể dùng `x & 1`.

In [3]:
for a, b in [(7, 4), (7, 3), (10, 2), (5, 8)]:
    print(f"{a} - {b} = {a - b}: lẻ? {(a - b) % 2 == 1} | a%2={a % 2}, b%2={b % 2}, khác nhau? {a % 2 != b % 2}")
print(-3 % 2, -3 & 1)

7 - 4 = 3: lẻ? True | a%2=1, b%2=0, khác nhau? True
7 - 3 = 4: lẻ? False | a%2=1, b%2=1, khác nhau? False
10 - 2 = 8: lẻ? False | a%2=0, b%2=0, khác nhau? False
5 - 8 = -3: lẻ? True | a%2=1, b%2=0, khác nhau? True
1 1


### 3.3 Đếm bằng mảng 2 phần tử hoặc `Counter`

`cnt = [1, 0]` nghĩa là: đã thấy 1 prefix chẵn (prefix rỗng), 0 prefix lẻ. `cnt[p % 2] += 1`.

In [4]:
cnt = [1, 0]
for p in [1, 4, 9]:
    cnt[p % 2] += 1
print("số prefix chẵn, lẻ:", cnt)
print(Counter(p % 2 for p in [0, 1, 4, 9]))

số prefix chẵn, lẻ: [2, 2]
Counter({0: 2, 1: 2})


### 3.4 Gán song song (tuple assignment)

`a, b = b + 1, a` tính **toàn bộ vế phải trước** rồi mới gán — tránh lỗi dùng giá trị vừa cập nhật (quan trọng cho Cách 3).

In [5]:
a, b = 2, 5
a, b = b + 1, a          # vế phải dùng a=2, b=5 cũ
print(a, b)
a, b = 2, 5
a = b + 1                # tuần tự: b dùng a MỚI
b = a
print(a, b, "<- khác!")

6 2
6 6 <- khác!


### 3.5 Modulo

Với `n = 10⁵`, số mảng con lên tới `5·10⁹` > `10⁹ + 7`. Python không tràn số; vẫn phải `% MOD` trước khi trả về.

In [6]:
n = 10**5
print("tổng số mảng con:", n * (n + 1) // 2, "> MOD?", n * (n + 1) // 2 > MOD)

tổng số mảng con: 5000050000 > MOD? True


## 4. Ý tưởng nền tảng / trực giác

Mỗi mảng con `arr[i..j]` ứng với **một cặp** prefix `(P[i], P[j+1])` với `i < j + 1`. Tổng lẻ ⇔ hai prefix khác tính chẵn lẻ.

Tính tay Ví dụ 1 `arr = [1, 3, 5]`:

| k | P[k] | chẵn/lẻ |
|---|---|---|
| 0 | 0 | chẵn |
| 1 | 1 | lẻ |
| 2 | 4 | chẵn |
| 3 | 9 | lẻ |

Có `E = 2` prefix chẵn (`P0, P2`) và `O = 2` prefix lẻ (`P1, P3`). Mỗi cặp (chẵn, lẻ) cho đúng một mảng con tổng lẻ → `E · O = 4` ✔.

**Đếm khi quét:** đi qua `P[1], P[2], …`; tại `P[k]` lẻ, số mảng con tổng lẻ **kết thúc tại `k−1`** bằng số prefix chẵn **trước đó**; tại `P[k]` chẵn thì bằng số prefix lẻ trước đó.

In [7]:
arr = [1, 3, 5]
P = [0] + list(accumulate(arr))
pairs = [(i, k) for i in range(len(P)) for k in range(i + 1, len(P)) if (P[k] - P[i]) % 2]
print("P =", P)
print("các cặp (i, k) khác chẵn lẻ:", pairs, "-> mảng con:", [arr[i:k] for i, k in pairs])

P = [0, 1, 4, 9]
các cặp (i, k) khác chẵn lẻ: [(0, 1), (0, 3), (1, 2), (2, 3)] -> mảng con: [[1], [1, 3, 5], [3], [5]]


## 5. Cách 1 — Brute Force / baseline

**Ý tưởng:** cố định điểm đầu `i`, cộng dồn `s` khi `j` tăng; mỗi khi `s` lẻ thì tăng đếm. (Bản ngây thơ hơn dùng `sum(arr[i:j+1])` là `O(n³)`.)

In [8]:
def brute(arr):
    n = len(arr)
    cnt = 0
    for i in range(n):
        s = 0
        for j in range(i, n):
            s += arr[j]
            if s % 2 == 1:
                cnt += 1
    return cnt % MOD

for arr, want in examples:
    print(arr, brute(arr), "kỳ vọng", want)

[1, 3, 5] 4 kỳ vọng 4
[2, 4, 6] 0 kỳ vọng 0
[1, 2, 3, 4, 5, 6, 7] 16 kỳ vọng 16


**Độ phức tạp:** Time `O(n²)` (`5·10⁹` bước với `n = 10⁵` — quá chậm), Space `O(1)`.

**Nút thắt:** ta xét từng cặp (đầu, cuối) riêng lẻ. Nhưng điều duy nhất quan trọng là **tính chẵn lẻ** của prefix → chỉ cần **đếm** bao nhiêu prefix chẵn/lẻ đã thấy.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** `sum(arr[i..j]) = P[j+1] − P[i]` lẻ ⇔ `P[j+1] % 2 ≠ P[i] % 2`.

Giữ `even` = số prefix chẵn đã thấy (bắt đầu `1` cho `P[0] = 0`), `odd` = số prefix lẻ đã thấy (bắt đầu `0`). Quét `j`:
- *nếu* prefix hiện tại **lẻ** *thì* `ans += even`, rồi `odd += 1`;
- *nếu* prefix hiện tại **chẵn** *thì* `ans += odd`, rồi `even += 1`.

**Cách nhìn DP:** gọi `O_i`/`E_i` = số mảng con **kết thúc tại `i`** có tổng lẻ/chẵn. Thêm `arr[i]`:
- `arr[i]` lẻ: mọi mảng con kết thúc tại `i−1` **đổi** tính chẵn lẻ, cộng mảng con `[arr[i]]` (lẻ): `O_i = E_{i−1} + 1`, `E_i = O_{i−1}`.
- `arr[i]` chẵn: giữ nguyên tính chẵn lẻ, `[arr[i]]` chẵn: `O_i = O_{i−1}`, `E_i = E_{i−1} + 1`.

**Công thức đóng:** đáp án `= E · O` với `E, O` là số prefix chẵn/lẻ trong `P[0..n]`.

## 7. Các cách tiếp theo

### Cách 2 — Đếm prefix chẵn/lẻ một lượt `O(n)`

In [9]:
def prefix_parity(arr):
    even, odd = 1, 0           # P[0] = 0 là chẵn
    s = ans = 0
    for x in arr:
        s += x
        if s % 2:
            ans += even
            odd += 1
        else:
            ans += odd
            even += 1
    return ans % MOD

for arr, want in examples:
    print(arr, prefix_parity(arr), "kỳ vọng", want)

[1, 3, 5] 4 kỳ vọng 4
[2, 4, 6] 0 kỳ vọng 0
[1, 2, 3, 4, 5, 6, 7] 16 kỳ vọng 16


Chỉ cần giữ **tính chẵn lẻ** của `s` (không cần giá trị thật): `p ^= x & 1`.

In [10]:
class SolutionPrefix:
    def numOfSubarrays(self, arr: list[int]) -> int:
        cnt = [1, 0]           # cnt[0] = số prefix chẵn, cnt[1] = số prefix lẻ
        p = ans = 0
        for x in arr:
            p ^= x & 1
            ans += cnt[p ^ 1]  # ghép với prefix KHÁC tính chẵn lẻ
            cnt[p] += 1
        return ans % (10**9 + 7)

print([SolutionPrefix().numOfSubarrays(a) for a, _ in examples])

[4, 0, 16]


**Độ phức tạp:** Time `O(n)`, Space `O(1)`.

### Cách 3 — DP: số mảng con kết thúc tại `i` có tổng lẻ / chẵn

In [11]:
def dp_end(arr):
    odd_end = even_end = 0
    ans = 0
    for x in arr:
        if x % 2:
            odd_end, even_end = even_end + 1, odd_end
        else:
            odd_end, even_end = odd_end, even_end + 1
        ans += odd_end
    return ans % MOD

for arr, want in examples:
    print(arr, dp_end(arr), "kỳ vọng", want)

[1, 3, 5] 4 kỳ vọng 4
[2, 4, 6] 0 kỳ vọng 0
[1, 2, 3, 4, 5, 6, 7] 16 kỳ vọng 16


In [12]:
class SolutionDP:
    def numOfSubarrays(self, arr: list[int]) -> int:
        MOD = 10**9 + 7
        odd = even = ans = 0
        for x in arr:
            if x & 1:
                odd, even = even + 1, odd
            else:
                even += 1
            ans = (ans + odd) % MOD
        return ans

print([SolutionDP().numOfSubarrays(a) for a, _ in examples])

[4, 0, 16]


**Độ phức tạp:** Time `O(n)`, Space `O(1)`. Ở đây lấy mod trong vòng lặp (thói quen tốt khi chuyển sang C++/Java).

### Cách 4 — Công thức đóng `E · O`

Đếm số prefix chẵn `E` và lẻ `O` trong `P[0..n]` (nhớ `P[0]`). Mỗi cặp khác tính chẵn lẻ ứng với đúng một mảng con tổng lẻ.

In [13]:
def closed_form(arr):
    parities = [0] + [p % 2 for p in accumulate(arr)]
    O = sum(parities)
    E = len(parities) - O
    return (E * O) % MOD

for arr, want in examples:
    print(arr, closed_form(arr), "kỳ vọng", want)

[1, 3, 5] 4 kỳ vọng 4
[2, 4, 6] 0 kỳ vọng 0
[1, 2, 3, 4, 5, 6, 7] 16 kỳ vọng 16


In [14]:
class SolutionClosed:
    def numOfSubarrays(self, arr: list[int]) -> int:
        odd = 0
        p = 0
        for x in arr:
            p ^= x & 1
            odd += p
        even = len(arr) + 1 - odd
        return odd * even % (10**9 + 7)

print([SolutionClosed().numOfSubarrays(a) for a, _ in examples])

[4, 0, 16]


**Độ phức tạp:** Time `O(n)`, Space `O(1)` (bản class) — gọn nhất và đẹp về mặt toán.

## 8. TRACE từng bước

Trace Cách 2 trên Ví dụ 3 `[1,2,3,4,5,6,7]`:

| j | x | prefix s | chẵn/lẻ | cộng | ans | (even, odd) sau |
|---|---|---|---|---|---|---|
| — | — | 0 | chẵn | — | 0 | (1, 0) |
| 0 | 1 | 1 | lẻ | +even=1 | 1 | (1, 1) |
| 1 | 2 | 3 | lẻ | +1 | 2 | (1, 2) |
| 2 | 3 | 6 | chẵn | +odd=2 | 4 | (2, 2) |
| 3 | 4 | 10 | chẵn | +2 | 6 | (3, 2) |
| 4 | 5 | 15 | lẻ | +3 | 9 | (3, 3) |
| 5 | 6 | 21 | lẻ | +3 | 12 | (3, 4) |
| 6 | 7 | 28 | chẵn | +4 | 16 | (4, 4) |

In [15]:
def trace_prefix(arr):
    even, odd = 1, 0
    s = ans = 0
    print(f"khởi tạo: even={even} odd={odd}")
    for j, x in enumerate(arr):
        s += x
        if s % 2:
            add = even
            ans += add
            odd += 1
            kind = "lẻ "
        else:
            add = odd
            ans += add
            even += 1
            kind = "chẵn"
        print(f"j={j} x={x} s={s:2d} ({kind}) cộng {add} -> ans={ans:2d} | even={even} odd={odd}")
    return ans % MOD

print("kết quả:", trace_prefix([1, 2, 3, 4, 5, 6, 7]))

khởi tạo: even=1 odd=0
j=0 x=1 s= 1 (lẻ ) cộng 1 -> ans= 1 | even=1 odd=1
j=1 x=2 s= 3 (lẻ ) cộng 1 -> ans= 2 | even=1 odd=2
j=2 x=3 s= 6 (chẵn) cộng 2 -> ans= 4 | even=2 odd=2
j=3 x=4 s=10 (chẵn) cộng 2 -> ans= 6 | even=3 odd=2
j=4 x=5 s=15 (lẻ ) cộng 3 -> ans= 9 | even=3 odd=3
j=5 x=6 s=21 (lẻ ) cộng 3 -> ans=12 | even=3 odd=4
j=6 x=7 s=28 (chẵn) cộng 4 -> ans=16 | even=4 odd=4
kết quả: 16


Trace Cách 3 (DP) trên cùng mảng — `odd_end`/`even_end` là số mảng con **kết thúc tại j** có tổng lẻ/chẵn:

In [16]:
def trace_dp(arr):
    odd_end = even_end = ans = 0
    for j, x in enumerate(arr):
        if x % 2:
            odd_end, even_end = even_end + 1, odd_end
            rule = "x lẻ : odd=even+1, even=odd"
        else:
            odd_end, even_end = odd_end, even_end + 1
            rule = "x chẵn: even+=1"
        ans += odd_end
        ends = [arr[i:j + 1] for i in range(j + 1) if sum(arr[i:j + 1]) % 2]
        print(f"j={j} x={x} [{rule}] odd_end={odd_end} even_end={even_end} ans={ans} | lẻ kết thúc tại j: {ends}")
    return ans

print("kết quả:", trace_dp([1, 2, 3, 4, 5, 6, 7]))

j=0 x=1 [x lẻ : odd=even+1, even=odd] odd_end=1 even_end=0 ans=1 | lẻ kết thúc tại j: [[1]]
j=1 x=2 [x chẵn: even+=1] odd_end=1 even_end=1 ans=2 | lẻ kết thúc tại j: [[1, 2]]
j=2 x=3 [x lẻ : odd=even+1, even=odd] odd_end=2 even_end=1 ans=4 | lẻ kết thúc tại j: [[2, 3], [3]]
j=3 x=4 [x chẵn: even+=1] odd_end=2 even_end=2 ans=6 | lẻ kết thúc tại j: [[2, 3, 4], [3, 4]]
j=4 x=5 [x lẻ : odd=even+1, even=odd] odd_end=3 even_end=2 ans=9 | lẻ kết thúc tại j: [[1, 2, 3, 4, 5], [4, 5], [5]]
j=5 x=6 [x chẵn: even+=1] odd_end=3 even_end=3 ans=12 | lẻ kết thúc tại j: [[1, 2, 3, 4, 5, 6], [4, 5, 6], [5, 6]]
j=6 x=7 [x lẻ : odd=even+1, even=odd] odd_end=4 even_end=3 ans=16 | lẻ kết thúc tại j: [[2, 3, 4, 5, 6, 7], [3, 4, 5, 6, 7], [6, 7], [7]]
kết quả: 16


## 9. Code nộp LeetCode

Bản chính: đếm prefix chẵn/lẻ (Cách 2).

In [17]:
class Solution:
    def numOfSubarrays(self, arr: list[int]) -> int:
        MOD = 10**9 + 7
        even, odd = 1, 0
        s = ans = 0
        for x in arr:
            s += x
            if s % 2:
                ans += even
                odd += 1
            else:
                ans += odd
                even += 1
        return ans % MOD

for arr, want in examples:
    assert Solution().numOfSubarrays(arr) == want
print("OK ví dụ đề")

OK ví dụ đề


## 10. Vì sao thuật toán đúng

**Song ánh:** mảng con `arr[i..j]` (với `0 ≤ i ≤ j < n`) ↔ cặp chỉ số prefix `(i, j+1)` với `0 ≤ i < j+1 ≤ n`. Tổng lẻ ⇔ `P[i]` và `P[j+1]` khác tính chẵn lẻ.

**Cách 2:** khi xử lý `P[k]` (`k = j + 1`), các biến `even/odd` đang đếm đúng số prefix chẵn/lẻ trong `P[0..k−1]` (bất biến, khởi tạo với `P[0] = 0` chẵn). Số cặp hợp lệ có đầu phải `k` = số prefix trước đó khác tính chẵn lẻ với `P[k]` → cộng đúng. Tổng trên mọi `k` = tổng số cặp hợp lệ.

**Cách 4:** số cặp không thứ tự (một chẵn, một lẻ) trong `P[0..n]` là `E·O`; mỗi cặp có đúng một thứ tự `i < k` → `E·O` mảng con.

**Cách 3:** quy nạp: thêm phần tử lẻ đổi tính chẵn lẻ mọi mảng con kết thúc tại `i−1` (khi nối thêm `arr[i]`), cộng mảng con đơn `[arr[i]]`; phần tử chẵn giữ nguyên. Tổng `Σ O_i` = số mảng con lẻ (phân hoạch theo điểm kết thúc). ∎

## 11. Phân tích Time / Space

`n = len(arr) ≤ 10⁵`.

| Cách | Time | Space |
|---|---|---|
| 1. Brute force | `O(n²)` | `O(1)` |
| 2. Đếm prefix chẵn/lẻ | `O(n)` | `O(1)` |
| 3. DP kết thúc tại i | `O(n)` | `O(1)` |
| 4. Công thức `E·O` | `O(n)` | `O(1)` (hoặc `O(n)` nếu tạo list) |

Không có khác biệt average/worst.

## 12. Bộ test

Ví dụ đề, edge case (một phần tử lẻ/chẵn, toàn lẻ, toàn chẵn, xen kẽ), random so với brute (seed cố định, đúng ràng buộc `1..100`), test lớn `n = 10⁵` để kiểm tra modulo.

In [18]:
METHODS = {
    "prefix": lambda a: Solution().numOfSubarrays(a),
    "prefix_xor": lambda a: SolutionPrefix().numOfSubarrays(a),
    "dp": lambda a: SolutionDP().numOfSubarrays(a),
    "closed": lambda a: SolutionClosed().numOfSubarrays(a),
}
edge = [([1], 1), ([2], 0), ([1, 1], 2), ([1, 1, 1], 4), ([2, 2, 2, 2], 0), ([1, 2, 1, 2], 6), ([100, 99], 2)]
for arr, want in examples + edge:
    for name, f in METHODS.items():
        got = f(arr)
        assert got == want, (name, arr, got, want)
print("ví dụ + edge: OK")

ví dụ + edge: OK


In [19]:
rng = random.Random(1524)
for _ in range(500):
    n = rng.randint(1, 40)
    hi = rng.choice([2, 10, 100])
    arr = [rng.randint(1, hi) for _ in range(n)]
    ref = brute(arr)
    for name, f in METHODS.items():
        assert f(arr) == ref, (name, arr)
print("500 mảng ngẫu nhiên: mọi cách khớp brute")

500 mảng ngẫu nhiên: mọi cách khớp brute


In [20]:
n = 10**5
alt = [1, 2] * (n // 2)                          # nhiều prefix lẻ và chẵn
rng2 = random.Random(5)
rand_big = [rng2.randint(1, 100) for _ in range(n)]
P = [0] + list(accumulate(alt))
O = sum(p % 2 for p in P)
exact = O * (len(P) - O)
print("đáp án thật cho mảng xen kẽ:", exact, "> MOD?", exact > MOD)
for name, f in METHODS.items():
    t = time.perf_counter()
    r1, r2 = f(alt), f(rand_big)
    print(f"{name:10s}: {r1} {r2} ({time.perf_counter() - t:.3f}s)")
    assert r1 == exact % MOD and r2 == METHODS["prefix"](rand_big)

đáp án thật cho mảng xen kẽ: 2500050000 > MOD? True
prefix    : 500049986 499994054 (0.015s)
prefix_xor: 500049986 499994054 (0.019s)


dp        : 500049986 499994054 (0.015s)


closed    : 500049986 499994054 (0.008s)


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên prefix rỗng (`even = 0` thay vì `1`)

Mất mọi mảng con **bắt đầu từ vị trí 0** có tổng lẻ.

In [21]:
def wrong_init(arr):
    even, odd = 0, 0                                  # SAI: phải là even = 1
    s = ans = 0
    for x in arr:
        s += x
        if s % 2:
            ans += even; odd += 1
        else:
            ans += odd; even += 1
    return ans % MOD

for arr, want in examples:
    print(arr, "| sai:", wrong_init(arr), "| đúng:", want)

[1, 3, 5] | sai: 2 | đúng: 4
[2, 4, 6] | sai: 0 | đúng: 0
[1, 2, 3, 4, 5, 6, 7] | sai: 12 | đúng: 16


### Lỗi 2 — DP gán tuần tự thay vì song song

In [22]:
def wrong_sequential(arr):
    odd_end = even_end = ans = 0
    for x in arr:
        if x % 2:
            odd_end = even_end + 1
            even_end = odd_end                       # SAI: dùng odd_end MỚI
        else:
            even_end += 1
        ans += odd_end
    return ans % MOD

for arr, want in examples:
    print(arr, "| sai:", wrong_sequential(arr), "| đúng:", want)

[1, 3, 5] | sai: 6 | đúng: 4
[2, 4, 6] | sai: 0 | đúng: 0
[1, 2, 3, 4, 5, 6, 7] | sai: 25 | đúng: 16


### Lỗi 3 — Chỉ đếm các prefix lẻ

`ans += 1` mỗi khi prefix lẻ chỉ đếm mảng con **bắt đầu tại 0**.

In [23]:
def wrong_only_prefix(arr):
    s = ans = 0
    for x in arr:
        s += x
        if s % 2:
            ans += 1                                 # SAI: bỏ mảng con bắt đầu giữa mảng
    return ans

for arr, want in examples:
    print(arr, "| sai:", wrong_only_prefix(arr), "| đúng:", want)

[1, 3, 5] | sai: 2 | đúng: 4
[2, 4, 6] | sai: 0 | đúng: 0
[1, 2, 3, 4, 5, 6, 7] | sai: 4 | đúng: 16


### Lỗi 4 — Quên modulo

In [24]:
def wrong_no_mod(arr):
    even, odd = 1, 0
    s = ans = 0
    for x in arr:
        s += x
        if s % 2:
            ans += even; odd += 1
        else:
            ans += odd; even += 1
    return ans                                       # SAI: thiếu % MOD

print("sai :", wrong_no_mod(alt))
print("đúng:", Solution().numOfSubarrays(alt))

sai : 2500050000


đúng: 500049986


### Lỗi 5 — Đếm "số phần tử lẻ" thay vì mảng con

Một mảng con có thể chứa nhiều số lẻ mà tổng vẫn chẵn (`[1, 3]`).

In [25]:
def wrong_count_odd_elements(arr):
    return sum(x % 2 for x in arr)                  # SAI: đếm phần tử, không phải mảng con

for arr, want in examples:
    print(arr, "| sai:", wrong_count_odd_elements(arr), "| đúng:", want)

[1, 3, 5] | sai: 3 | đúng: 4
[2, 4, 6] | sai: 0 | đúng: 0
[1, 2, 3, 4, 5, 6, 7] | sai: 4 | đúng: 16


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Đếm mảng con có tổng thoả điều kiện về số dư"** → prefix sum + **đếm số dư** đã thấy (hash map / mảng nhỏ), luôn khởi tạo với prefix rỗng.
- Chẵn/lẻ là trường hợp `k = 2` của "tổng chia hết cho k" / "tổng ≡ r (mod k)".
- Khi chỉ có 2 trạng thái → có công thức đóng `E·O`.
- DP "số mảng con kết thúc tại i có tính chất X" là cách nhìn thay thế, rất mạnh với các bài tích (LC 1567: tích dương dài nhất).
- Đề có "modulo 10⁹+7" ⇒ kiểm tra trường hợp đáp án ~`n²/4`.

## 15. Biến thể của bài

1. **LC 974 – Subarray Sums Divisible by K:** đếm số dư prefix `mod k`, cộng `cnt[r]`.
2. **LC 560 – Subarray Sum Equals K:** hash map giá trị prefix.
3. **LC 1248 – Count Number of Nice Subarrays:** đúng `k` số lẻ → prefix số lượng số lẻ.
4. **LC 523 – Continuous Subarray Sum:** độ dài ≥ 2, chia hết cho `k`.
5. **Tổng chẵn:** `C(E, 2) + C(O, 2)`.
6. **Tích lẻ:** mảng con chỉ gồm số lẻ → đếm nhóm số lẻ liên tiếp `L(L+1)/2`.

In [26]:
# Minh hoạ biến thể 5: số mảng con tổng chẵn = C(E,2) + C(O,2); tổng hai loại = n(n+1)/2
for arr, _ in examples:
    P = [0] + list(accumulate(arr))
    O = sum(p % 2 for p in P); E = len(P) - O
    even_cnt = E * (E - 1) // 2 + O * (O - 1) // 2
    n = len(arr)
    print(arr, "lẻ:", E * O, "| chẵn:", even_cnt, "| tổng:", E * O + even_cnt, "= n(n+1)/2 =", n * (n + 1) // 2)

[1, 3, 5] lẻ: 4 | chẵn: 2 | tổng: 6 = n(n+1)/2 = 6
[2, 4, 6] lẻ: 0 | chẵn: 6 | tổng: 6 = n(n+1)/2 = 6
[1, 2, 3, 4, 5, 6, 7] lẻ: 16 | chẵn: 12 | tổng: 28 = n(n+1)/2 = 28


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force | Mọi cặp (đầu, cuối), tổng chạy | `O(n²)` | `O(1)` | Đối chiếu |
| 2. Đếm prefix chẵn/lẻ | Ghép với prefix khác tính chẵn lẻ | `O(n)` | `O(1)` | Mặc định, tổng quát hoá dễ (mod k) |
| 3. DP kết thúc tại i | `O_i, E_i` đổi chỗ khi gặp số lẻ | `O(n)` | `O(1)` | Tư duy DP, bài tích/dấu |
| 4. Công thức `E·O` | Đếm cặp khác tính chẵn lẻ | `O(n)` | `O(1)` | Ngắn gọn nhất |

## 17. Bài tập vận dụng

### Bài 1 (đoán output)
`Solution().numOfSubarrays([1])`, `([2])`, `([1, 1])`, `([7, 7, 7, 7])` trả về gì?

### Bài 2 (tính tay)
Với `arr = [2, 3, 2, 3]`: liệt kê các prefix, đếm `E`, `O` và tính đáp án.

### Bài 3 (điền TODO)
Viết `count_even_sum(arr)`: số mảng con có **tổng chẵn** (mod 10⁹+7), `O(n)`.

### Bài 4 (tự viết lại)
Tự viết lại Cách 3 (DP) mà không nhìn lại: `num_odd_dp(arr)`.

### Bài 5 (mini challenge)
LC 974: `subarrays_div_by_k(nums, k)` — số mảng con có tổng chia hết cho `k` (`nums` có thể âm).

In [27]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def count_even_sum(arr):
    # TODO: ghép prefix CÙNG tính chẵn lẻ: ans += cnt[p]; nhớ cnt = [1, 0]
    pass

# Bài 4 — TODO
def num_odd_dp(arr):
    # TODO: odd, even = even + 1, odd nếu x lẻ; even += 1 nếu x chẵn; ans += odd
    pass

# Bài 5 — TODO
def subarrays_div_by_k(nums, k):
    # TODO: cnt = {0: 1}; r = (r + x) % k; ans += cnt.get(r, 0); cnt[r] += 1
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) `[1]` có 1 mảng con tổng lẻ. (2) `[1,1]`: `[1],[1]` lẻ, `[1,1]` chẵn. (3) `[7,7,7,7]`: prefix `0,7,14,21,28` → `E = 3, O = 2` → 6.

**Bài 2.** (1) `P = 0, 2, 5, 7, 10`. (2) `E = 3` (0, 2, 10), `O = 2` (5, 7). (3) `6`.

**Bài 3.** (1) Tổng chẵn ⇔ prefix cùng tính chẵn lẻ. (2) Trước khi tăng `cnt[p]`, cộng `cnt[p]`. (3) Kiểm tra: lẻ + chẵn = `n(n+1)/2`.

**Bài 4.** (1) Hai biến `odd`, `even`. (2) Gán song song khi gặp số lẻ. (3) `ans += odd` mỗi bước.

**Bài 5.** (1) Python `%` trả số dư không âm kể cả số âm. (2) `cnt[0] = 1` cho prefix rỗng. (3) Cộng `cnt[r]` trước khi tăng.

## 19. Đáp án tham khảo cho bài tập

In [28]:
# Bài 1, 2
r1 = [Solution().numOfSubarrays(a) for a in ([1], [2], [1, 1], [7, 7, 7, 7])]
P = [0] + list(accumulate([2, 3, 2, 3]))
O = sum(p % 2 for p in P); E = len(P) - O
print("Bài 1:", r1, "| Bài 2: P =", P, "E =", E, "O =", O, "->", E * O)
assert r1 == [1, 0, 2, 6] and E * O == Solution().numOfSubarrays([2, 3, 2, 3]) == 6

Bài 1: [1, 0, 2, 6] | Bài 2: P = [0, 2, 5, 7, 10] E = 3 O = 2 -> 6


In [29]:
# Bài 3
def count_even_sum_sol(arr):
    cnt = [1, 0]
    p = ans = 0
    for x in arr:
        p ^= x & 1
        ans += cnt[p]
        cnt[p] += 1
    return ans % MOD

# Bài 4
def num_odd_dp_sol(arr):
    odd = even = ans = 0
    for x in arr:
        if x % 2:
            odd, even = even + 1, odd
        else:
            even += 1
        ans += odd
    return ans % MOD

rng = random.Random(4)
for _ in range(300):
    arr = [rng.randint(1, 100) for _ in range(rng.randint(1, 30))]
    n = len(arr)
    brute_even = sum(1 for i in range(n) for j in range(i, n) if sum(arr[i:j + 1]) % 2 == 0)
    assert count_even_sum_sol(arr) == brute_even
    assert num_odd_dp_sol(arr) == brute(arr)
    assert count_even_sum_sol(arr) + num_odd_dp_sol(arr) == n * (n + 1) // 2
print("Bài 3, 4: OK")

Bài 3, 4: OK


In [30]:
# Bài 5
def subarrays_div_by_k_sol(nums, k):
    cnt = {0: 1}
    r = ans = 0
    for x in nums:
        r = (r + x) % k
        ans += cnt.get(r, 0)
        cnt[r] = cnt.get(r, 0) + 1
    return ans

print("Bài 5:", subarrays_div_by_k_sol([4, 5, 0, -2, -3, 1], 5), "(ví dụ LC 974 -> 7)")
assert subarrays_div_by_k_sol([4, 5, 0, -2, -3, 1], 5) == 7
assert subarrays_div_by_k_sol([5], 9) == 0
for _ in range(300):
    nums = [rng.randint(-20, 20) for _ in range(rng.randint(1, 25))]
    k = rng.randint(2, 9)
    n = len(nums)
    ref = sum(1 for i in range(n) for j in range(i, n) if sum(nums[i:j + 1]) % k == 0)
    assert subarrays_div_by_k_sol(nums, k) == ref
print("Đáp án Bài 1–5: mọi assert đều qua")

Bài 5: 7 (ví dụ LC 974 -> 7)
Đáp án Bài 1–5: mọi assert đều qua


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "đếm mảng con theo số dư của tổng":**

```python
cnt = {0: 1}                 # prefix rỗng!
r = ans = 0
for x in arr:
    r = (r + x) % k
    ans += cnt.get(target_partner(r), 0)   # chẵn/lẻ: partner = 1 - r
    cnt[r] = cnt.get(r, 0) + 1
return ans % MOD
```

**Checklist trước khi submit:**
- [ ] Khởi tạo `even = 1` (prefix rỗng).
- [ ] Prefix lẻ ghép với **chẵn**, prefix chẵn ghép với **lẻ**.
- [ ] DP: gán song song `odd, even = even + 1, odd`.
- [ ] `% (10**9 + 7)`.
- [ ] Đúng chữ ký: `numOfSubarrays(self, arr: list[int]) -> int`.